# NeuroX v4 - Fast GPU Training (Google Colab T4)

## Train ALL 17 models in 5-15 minutes instead of 12+ hours!

**How it works:**
- Colab gives you a free NVIDIA T4 GPU (8.1 TFLOPS vs ~0.1 on CPU)
- GPU batch size = 256 (vs 32 on CPU) = 8x fewer gradient steps per epoch
- Total speedup: ~80-100x faster than your Windows trading PC

**Instructions:**
1. Make sure GPU is enabled: Runtime -> Change runtime type -> T4 GPU
2. Click "Runtime -> Run all" (or run cells one by one)
3. Wait ~10 minutes for all 17 models to train
4. Download the checkpoints.zip file at the end
5. Unzip into your trading PC: neurox_v4/checkpoints/

**No CSV upload needed** - data is downloaded from Yahoo Finance automatically!

**IMPORTANT:** This notebook clones the `fast-gpu-training` branch which contains
the NeuroX v4 code and the `train_gpu_fast.py` script. The `main` branch does NOT
have these files.


In [ ]:
#@title Step 0: Verify GPU is available (MUST show T4)
import subprocess
import sys

print("=" * 60)
print("  STEP 0: GPU Verification")
print("=" * 60)

try:
    import torch
except ImportError:
    print("PyTorch not found, will be installed in Step 1.")
    print("For now, checking nvidia-smi...")
    result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
    if result.returncode == 0:
        print(result.stdout)
        print("\nGPU detected via nvidia-smi. Proceeding.")
    else:
        raise RuntimeError(
            "No GPU detected! Go to: Runtime -> Change runtime type -> T4 GPU\n"
            "Then click 'Runtime -> Restart and run all'"
        )
else:
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
        print(f"  GPU: {gpu_name}")
        print(f"  VRAM: {vram:.1f} GB")
        print(f"  CUDA: {torch.version.cuda}")
        print()
        print("  GPU is ready! Training will be FAST.")
    else:
        raise RuntimeError(
            "No GPU detected! Go to: Runtime -> Change runtime type -> T4 GPU\n"
            "Then click 'Runtime -> Restart and run all'"
        )

print("=" * 60)


In [ ]:
#@title Step 1: Clone repo (fast-gpu-training branch) & Install dependencies (~2 min)
import os
import subprocess

print("=" * 60)
print("  STEP 1: Clone & Install")
print("=" * 60)

REPO_URL = "https://github.com/gagandocx/Claude.git"
BRANCH = "fast-gpu-training"
CLONE_DIR = "/content/Claude"
WORK_DIR = "/content/Claude/NeuroX/neurox_v4"

# ---- Clone or update the repo ----
if not os.path.exists(CLONE_DIR):
    print(f"\n[1/3] Cloning branch '{BRANCH}' from {REPO_URL}...")
    result = subprocess.run(
        ["git", "clone", "-b", BRANCH, "--single-branch", REPO_URL, CLONE_DIR],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f"STDERR: {result.stderr}")
        raise RuntimeError(f"git clone failed! Error: {result.stderr}")
    print("  Repo cloned successfully!")
else:
    print(f"\n[1/3] Repo already exists. Pulling latest from '{BRANCH}'...")
    # Make sure we're on the right branch
    subprocess.run(["git", "checkout", BRANCH], cwd=CLONE_DIR, capture_output=True)
    subprocess.run(["git", "pull", "origin", BRANCH], cwd=CLONE_DIR, capture_output=True)
    print("  Repo updated!")

# ---- Verify critical files exist ----
print("\n[2/3] Verifying project files...")
critical_files = [
    os.path.join(WORK_DIR, "train_gpu_fast.py"),
    os.path.join(WORK_DIR, "config", "settings.py"),
    os.path.join(WORK_DIR, "data", "market_data.py"),
    os.path.join(WORK_DIR, "models"),
]
missing = []
for f in critical_files:
    exists = os.path.exists(f)
    status = "OK" if exists else "MISSING"
    print(f"  [{status}] {f}")
    if not exists:
        missing.append(f)

if missing:
    print("\n" + "!" * 60)
    print("  CRITICAL ERROR: Required files are missing!")
    print("  This means the branch clone failed or the branch is wrong.")
    print("\n  Attempting to fix by re-cloning...")
    import shutil
    shutil.rmtree(CLONE_DIR, ignore_errors=True)
    result = subprocess.run(
        ["git", "clone", "-b", BRANCH, "--single-branch", REPO_URL, CLONE_DIR],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        raise RuntimeError(
            f"Failed to clone branch '{BRANCH}'. Error: {result.stderr}\n"
            f"Make sure the branch exists at: {REPO_URL}"
        )
    # Re-verify
    still_missing = [f for f in critical_files if not os.path.exists(f)]
    if still_missing:
        raise RuntimeError(
            f"Files still missing after re-clone: {still_missing}\n"
            f"The branch '{BRANCH}' may not contain the NeuroX code."
        )
    print("  Fixed! All files present after re-clone.")

# ---- Install Python dependencies ----
print("\n[3/3] Installing Python dependencies...")
print("  (torch with CUDA, scikit-learn, lightgbm, catboost, xgboost, etc.)")

deps = [
    "torch",
    "scikit-learn==1.9.0",
    "lightgbm",
    "catboost",
    "xgboost",
    "yfinance",
    "ta",
    "pandas",
    "numpy",
    "tqdm",
    "joblib",
    "hmmlearn",
    "scipy",
]

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + deps,
    capture_output=True, text=True
)
if result.returncode != 0:
    print(f"  pip install output: {result.stdout}")
    print(f"  pip install errors: {result.stderr}")
    # Don't fail hard - some packages might already be installed
    print("  WARNING: Some packages may have failed. Will try to continue.")
else:
    print("  All dependencies installed successfully!")

# ---- Final verification ----
print("\n" + "=" * 60)
print("  Setup complete!")
print(f"  Working directory: {WORK_DIR}")
print(f"  Branch: {BRANCH}")
print("  Ready to train.")
print("=" * 60)


In [ ]:
#@title Step 2: Train ALL 17 models on GPU (~5-15 min)
import os
import subprocess
import sys

print("=" * 60)
print("  STEP 2: Training ALL 17 Models")
print("=" * 60)

WORK_DIR = "/content/Claude/NeuroX/neurox_v4"

# Verify we're in the right place
train_script = os.path.join(WORK_DIR, "train_gpu_fast.py")
if not os.path.exists(train_script):
    raise RuntimeError(
        f"train_gpu_fast.py not found at {train_script}!\n"
        "Did Step 1 complete successfully? Re-run Step 1 first."
    )

print(f"\n  Script: {train_script}")
print(f"  Working dir: {WORK_DIR}")
print(f"  This will take ~5-15 minutes on a T4 GPU.")
print(f"  Watch the progress bars below...\n")

# Run training with real-time output
os.chdir(WORK_DIR)

# Use os.system for real-time output in Colab (subprocess captures it)
exit_code = os.system(f"cd {WORK_DIR} && {sys.executable} train_gpu_fast.py")

if exit_code != 0:
    print("\n" + "!" * 60)
    print("  Training failed!")
    print("  Common fixes:")
    print("  1. Make sure GPU is enabled (Runtime -> Change runtime type -> T4)")
    print("  2. Restart runtime (Runtime -> Restart runtime) and re-run all cells")
    print("  3. Check if Colab session ran out of RAM (try again in a new session)")
    print("!" * 60)
    raise RuntimeError("Training script exited with errors. See output above.")
else:
    print("\n" + "=" * 60)
    print("  Training COMPLETE!")
    print("  All models saved to: /content/Claude/NeuroX/neurox_v4/checkpoints/")
    print("=" * 60)


In [ ]:
#@title Step 3: Download trained checkpoints to your PC
import os
import shutil

print("=" * 60)
print("  STEP 3: Package & Download Checkpoints")
print("=" * 60)

checkpoint_dir = "/content/Claude/NeuroX/neurox_v4/checkpoints"

if not os.path.exists(checkpoint_dir):
    raise RuntimeError(
        f"Checkpoints directory not found at {checkpoint_dir}!\n"
        "Training may have failed. Re-run Step 2."
    )

# List all saved checkpoint files
files_list = sorted(os.listdir(checkpoint_dir))
if not files_list:
    raise RuntimeError(
        "Checkpoints directory is empty! Training did not save any models.\n"
        "Re-run Step 2 and check for errors."
    )

print("\nSaved checkpoints:")
total_size = 0
for f in files_list:
    fpath = os.path.join(checkpoint_dir, f)
    size = os.path.getsize(fpath)
    total_size += size
    print(f"  {f:<40} {size / 1024:.0f} KB")

print(f"\n  Total: {len(files_list)} files, {total_size / (1024*1024):.1f} MB")

# Create zip archive
zip_path = "/content/neurox_checkpoints"
print(f"\nCreating zip archive...")
shutil.make_archive(zip_path, "zip", checkpoint_dir)
zip_size = os.path.getsize(f"{zip_path}.zip") / (1024 * 1024)
print(f"  Archive: {zip_path}.zip ({zip_size:.1f} MB)")

# Download
print("\nStarting download...")
try:
    from google.colab import files
    files.download(f"{zip_path}.zip")
    print("\n  Download started! Check your browser downloads.")
except ImportError:
    print("  Not running in Colab - skipping auto-download.")
    print(f"  Zip file saved at: {zip_path}.zip")
except Exception as e:
    print(f"  Auto-download failed: {e}")
    print("  Use the 'Alternative: Save to Google Drive' cell below instead.")

print("\n" + "=" * 60)
print("  DONE! After downloading, unzip into your trading PC:")
print("  neurox_v4/checkpoints/")
print("=" * 60)


## Alternative: Save to Google Drive (if download fails)

If the direct download doesn't work (Colab can be flaky with large downloads),
use Google Drive instead:


In [ ]:
#@title Alternative: Save to Google Drive
import os
import shutil

try:
    from google.colab import drive
    drive.mount("/content/drive")

    src = "/content/Claude/NeuroX/neurox_v4/checkpoints"
    dst = "/content/drive/MyDrive/neurox_v4_checkpoints"

    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.copytree(src, dst)

    print(f"Checkpoints saved to Google Drive: {dst}")
    print()
    print("On your trading PC:")
    print("  1. Open Google Drive in browser")
    print("  2. Download the neurox_v4_checkpoints folder")
    print("  3. Copy contents into: neurox_v4/checkpoints/")
except ImportError:
    print("Not running in Colab. Cannot mount Google Drive.")
except Exception as e:
    print(f"Google Drive save failed: {e}")
    print("Try running this cell again, or use the direct download above.")


## Troubleshooting

**"No GPU detected"**
- Go to Runtime -> Change runtime type -> Select T4 GPU -> Save
- Then Runtime -> Restart and run all

**"train_gpu_fast.py not found"**
- The clone may have used the wrong branch. Delete and re-clone:
  ```python
  !rm -rf /content/Claude
  ```
- Then re-run from Step 1

**"ModuleNotFoundError"**
- Re-run Step 1 to install dependencies
- If `config.settings` or `data.market_data` is missing, the branch is wrong

**Training runs out of memory**
- Restart runtime: Runtime -> Restart runtime
- Re-run all cells from the beginning

**Download doesn't work**
- Use the "Save to Google Drive" cell instead

## How to retrain (weekly)

Run this notebook once a week to keep your models fresh with latest market data:
1. Open this notebook in Colab
2. Runtime -> Run all
3. Download new checkpoints
4. Replace checkpoints/ folder on your trading PC
5. Restart your EA

The models will train on the latest 7 days of 1-minute data, 60 days of 15-minute
data, and 2 years of hourly data from Yahoo Finance, ensuring they stay adapted to
current market conditions.
